# Notebook 25. Continuous Spell Evolution and Onset Timing

This notebook is the broader time-evolution follow-on to `Notebooks 22–23`.

It is designed for the problem we uncovered in the peak-event method:

- some offshore-to-coastal or coastal-to-offshore evolution may happen between catalog peaks
- later stages can be missed if they never became their own event peak
- we still want those stages to count when they are visible in the underlying ERA5 fields

So this notebook:

- builds broader spells from catalog chronology only
- pads each spell window by `+/- 48 h`
- computes continuous polygon and coastal-only moisture/divergence metrics
- applies candidate threshold profiles through time with a persistence rule
- estimates onset order and lag from the continuous spell windows instead of peak-event order alone


In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git"
BRANCH = os.environ.get("JPCZ_CATALOG_BRANCH", "codex/notebook16-pcolormesh")
REPO_DIR = "/content/JPCZcatalog"
FORCE_REFRESH_REPO = False
PERSIST_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/JPCZcatalog_outputs"

if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print("Persistent output dir:", DRIVE_OUTPUT_DIR)

os.chdir("/content")

def clone_repo_branch():
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
        text=True,
        capture_output=True,
    )
    print(proc.stdout)
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{proc.stderr}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
        check=True,
    )

def sync_repo_branch():
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)


if FORCE_REFRESH_REPO and os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Removed existing repo clone:", REPO_DIR)

if not os.path.exists(REPO_DIR):
    clone_repo_branch()
else:
    print("Using existing repo clone:", REPO_DIR)

try:
    sync_repo_branch()
except subprocess.CalledProcessError:
    print("Existing clone could not switch branches cleanly. Re-cloning target branch.")
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    clone_repo_branch()
    sync_repo_branch()

os.chdir(REPO_DIR)
src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

active_branch = subprocess.run(["git", "-C", REPO_DIR, "branch", "--show-current"], text=True, capture_output=True, check=True).stdout.strip()
print("Working directory:", os.getcwd())
print("Runtime repo branch:", active_branch)


In [ ]:
from pathlib import Path
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from jpcz_catalog.analysis import add_japan_local_time_columns
from jpcz_catalog.config import JPCZ_POLYGON_VERTICES, OBJECTIVE_SUBTYPE_DOMAIN
from jpcz_catalog.era5 import open_arco_era5, subset_era5_window
from jpcz_catalog.objective_regimes import summarize_padded_spell_windows
from jpcz_catalog.spell_evolution import (
    apply_continuous_regime_thresholds,
    compute_window_regional_metric_timeseries,
    prepare_objective_region_weights,
    summarize_continuous_spell,
)

OBJECTIVE_EXPORT_DIR = Path("outputs/verification/objective_coastal_box_regimes")
VERIFICATION_DIR = Path("outputs/verification")
TIMING_EXPORT_DIR = Path("outputs/verification/objective_regime_timing_and_impact")
TIMING_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR = Path("outputs/verification/objective_regime_timing_and_impact_plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

OBJECTIVE_EVENT_METRICS_PATH = OBJECTIVE_EXPORT_DIR / "objective_coastal_box_event_metrics.csv"
THRESHOLD_SWEEP_INPUT_PATH = OBJECTIVE_EXPORT_DIR / "objective_coastal_box_threshold_sensitivity.csv"
POSITION_INTENSITY_PATH = VERIFICATION_DIR / "jpcz_catalog_ndjf_merged_12h_position_intensity.csv"

ERA5_TIME_CHUNK = 48
FORCE_REBUILD_CONTINUOUS_SPELLS = False
CONTINUOUS_SPELL_GAP_HOURS = 36
CONTINUOUS_PADDING_HOURS = 48
CONTINUOUS_TIME_STEP_HOURS = 6
CONTINUOUS_PERSISTENCE_STEPS = 2
CONTINUOUS_PROGRESS_EVERY_SPELLS = 10
CONTINUOUS_SPELL_LIMIT = None
CONTINUOUS_PROFILE_FILTER = None
CONTINUOUS_PLOT_PROFILE = "balanced_looser"
CONTINUOUS_PLOT_SPELL_ID = None

COASTAL_STRIP_VERTICES = (
    (133.05, 35.55),
    (136.05, 35.55),
    (139.55, 39.00),
    (139.55, 42.55),
)

CONTINUOUS_CANDIDATE_THRESHOLDS_PATH = TIMING_EXPORT_DIR / f"objective_regime_continuous_candidate_thresholds_gap{CONTINUOUS_SPELL_GAP_HOURS:02d}h_pad{CONTINUOUS_PADDING_HOURS:02d}h.csv"
CONTINUOUS_SPELL_EVENT_PATH = TIMING_EXPORT_DIR / f"objective_regime_continuous_spell_events_gap{CONTINUOUS_SPELL_GAP_HOURS:02d}h_pad{CONTINUOUS_PADDING_HOURS:02d}h.csv"
CONTINUOUS_SPELL_WINDOW_PATH = TIMING_EXPORT_DIR / f"objective_regime_continuous_spell_windows_gap{CONTINUOUS_SPELL_GAP_HOURS:02d}h_pad{CONTINUOUS_PADDING_HOURS:02d}h.csv"
CONTINUOUS_TIMESERIES_PATH = TIMING_EXPORT_DIR / f"objective_regime_continuous_spell_timeseries_gap{CONTINUOUS_SPELL_GAP_HOURS:02d}h_pad{CONTINUOUS_PADDING_HOURS:02d}h.csv"
CONTINUOUS_SUMMARY_PATH = TIMING_EXPORT_DIR / f"objective_regime_continuous_spell_summary_gap{CONTINUOUS_SPELL_GAP_HOURS:02d}h_pad{CONTINUOUS_PADDING_HOURS:02d}h.csv"
CONTINUOUS_PROFILE_SUMMARY_PATH = TIMING_EXPORT_DIR / f"objective_regime_continuous_profile_summary_gap{CONTINUOUS_SPELL_GAP_HOURS:02d}h_pad{CONTINUOUS_PADDING_HOURS:02d}h.csv"
CONTINUOUS_EXAMPLES_PATH = TIMING_EXPORT_DIR / f"objective_regime_continuous_spell_examples_gap{CONTINUOUS_SPELL_GAP_HOURS:02d}h_pad{CONTINUOUS_PADDING_HOURS:02d}h.csv"
CONTINUOUS_PLOT_PATH = PLOT_DIR / f"objective_regime_continuous_spell_profile_{CONTINUOUS_PLOT_PROFILE}_gap{CONTINUOUS_SPELL_GAP_HOURS:02d}h_pad{CONTINUOUS_PADDING_HOURS:02d}h.png"


def maybe_copy_to_drive(path: Path, *, verbose: bool = True):
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return None
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if path.is_file():
        shutil.copy2(path, drive_path)
        if verbose:
            print("Copied to Drive:", drive_path)
        return drive_path
    return None


def restore_from_drive_cache(path: Path) -> bool:
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return False
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if not drive_path.exists():
        return False
    path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_path, path)
    print("Restored from Drive cache:", drive_path)
    return True


In [ ]:
paths_to_restore = [
    OBJECTIVE_EVENT_METRICS_PATH,
    THRESHOLD_SWEEP_INPUT_PATH,
    POSITION_INTENSITY_PATH,
]
for path in paths_to_restore:
    if not path.exists():
        restore_from_drive_cache(path)

if not OBJECTIVE_EVENT_METRICS_PATH.exists():
    raise RuntimeError("Run Notebook 22 first so the objective event-label table exists.")
if not THRESHOLD_SWEEP_INPUT_PATH.exists():
    raise RuntimeError("Run Notebook 22 first so the full threshold-sensitivity table exists.")
if not POSITION_INTENSITY_PATH.exists():
    raise RuntimeError("Run Notebook 05 first so the merged 12 h position/intensity catalog exists.")

objective_df = pd.read_csv(
    OBJECTIVE_EVENT_METRICS_PATH,
    parse_dates=[column for column in ["event_start", "event_end", "event_peak", "event_peak_jst"] if column in pd.read_csv(OBJECTIVE_EVENT_METRICS_PATH, nrows=0).columns],
)
objective_df = add_japan_local_time_columns(objective_df)
if "event_index" not in objective_df.columns:
    objective_df["event_index"] = objective_df.index.astype(int)

threshold_sweep_input_df = pd.read_csv(THRESHOLD_SWEEP_INPUT_PATH)
threshold_column_aliases = {
    "is_default_setting": "default_setting",
    "polygon_qflux_min_quantile": "polygon_qflux_q",
    "polygon_qflux_min_value": "polygon_qflux_min",
    "coastal_qflux_split_quantile": "coastal_qflux_q",
    "coastal_qflux_split_value": "coastal_qflux_split",
    "polygon_div_max_quantile": "polygon_div_q",
    "polygon_div_max_value": "polygon_div_max",
    "coastal_div_max_quantile": "coastal_div_q",
    "coastal_div_max_value": "coastal_div_max",
    "offshore_jpcz_core_n": "offshore_n",
    "coastal_interaction_n": "coastal_n",
    "mixed_transition_n": "mixed_n",
    "weak_or_unclear_n": "weak_n",
    "sorted_event_count": "sorted_n",
    "sorted_fraction_of_201": "sorted_fraction",
}
threshold_sweep_input_df = threshold_sweep_input_df.rename(
    columns={column: alias for column, alias in threshold_column_aliases.items() if column in threshold_sweep_input_df.columns}
)
for column in list(threshold_sweep_input_df.columns):
    if column.startswith("polygon_qflux_min ["):
        threshold_sweep_input_df = threshold_sweep_input_df.rename(columns={column: "polygon_qflux_min"})
    elif column.startswith("coastal_qflux_split ["):
        threshold_sweep_input_df = threshold_sweep_input_df.rename(columns={column: "coastal_qflux_split"})
    elif column.startswith("polygon_div_max ["):
        threshold_sweep_input_df = threshold_sweep_input_df.rename(columns={column: "polygon_div_max"})
    elif column.startswith("coastal_div_max ["):
        threshold_sweep_input_df = threshold_sweep_input_df.rename(columns={column: "coastal_div_max"})

required_threshold_columns = [
    "default_setting",
    "polygon_qflux_q",
    "polygon_qflux_min",
    "coastal_qflux_q",
    "coastal_qflux_split",
    "polygon_div_q",
    "polygon_div_max",
    "coastal_div_q",
    "coastal_div_max",
    "offshore_n",
    "coastal_n",
    "mixed_n",
    "weak_n",
    "sorted_n",
    "sorted_fraction",
]
missing_threshold_columns = [column for column in required_threshold_columns if column not in threshold_sweep_input_df.columns]
if missing_threshold_columns:
    raise RuntimeError(
        "Notebook 22 threshold-sensitivity output is missing required columns even after normalization: "
        f"{missing_threshold_columns}. Available columns: {threshold_sweep_input_df.columns.tolist()}"
    )
threshold_sweep_input_df["default_setting"] = (
    threshold_sweep_input_df["default_setting"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({"true": True, "false": False})
    .fillna(False)
    .astype(bool)
)

position_df = pd.read_csv(
    POSITION_INTENSITY_PATH,
    parse_dates=["event_start", "event_end", "event_peak"],
)
position_df = add_japan_local_time_columns(position_df)
position_df["event_index"] = position_df.index.astype(int)

position_columns = [
    "event_index",
    "event_start",
    "event_end",
    "duration_hours",
    "episode_span_hours",
    "peak_max_convergence_1e5_s-1",
    "peak_max_convergence_lat",
    "peak_max_convergence_lon",
    "candidate_peak_convergence_percentile",
    "convergence_intensity_auto",
    "lat_position_group_auto",
    "lon_position_group_auto",
    "monsoon_type",
    "shinoda_class",
]
position_merge_df = position_df[[column for column in position_columns if column in position_df.columns]].copy()
event_df = objective_df.merge(position_merge_df, on="event_index", how="left", suffixes=("", "_position"))
for column in ["event_start", "event_end"]:
    fallback_column = f"{column}_position"
    if column in event_df.columns and fallback_column in event_df.columns:
        event_df[column] = event_df[column].fillna(event_df[fallback_column])
        event_df = event_df.drop(columns=[fallback_column])

print("Notebook 25 setup")
print(f"- Objective-labeled events available: {len(event_df)}")
print(f"- Threshold combinations available from Notebook 22: {len(threshold_sweep_input_df)}")
print(f"- Continuous spell gap: {CONTINUOUS_SPELL_GAP_HOURS} h")
print(f"- Continuous analysis padding: +/- {CONTINUOUS_PADDING_HOURS} h")
print(f"- Continuous time-step interval: {CONTINUOUS_TIME_STEP_HOURS} h")
print(f"- Persistence rule: {CONTINUOUS_PERSISTENCE_STEPS} consecutive time steps")


# Continuous Spell Screen

This is the heavy-lift cell, so it now has a few safety valves:

- `CONTINUOUS_SPELL_LIMIT` to test only the first N spells
- `CONTINUOUS_PROFILE_FILTER` to run only one threshold profile first
- progress prints every 10 spells
- periodic CSV checkpoint overwrites so a long run is less opaque


In [ ]:
required_globals = [
    "event_df",
    "threshold_sweep_input_df",
    "restore_from_drive_cache",
    "maybe_copy_to_drive",
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(f"Run the Notebook 23 setup cells first. Missing globals: {missing_globals}")
continuous_paths = [
    CONTINUOUS_CANDIDATE_THRESHOLDS_PATH,
    CONTINUOUS_SPELL_EVENT_PATH,
    CONTINUOUS_SPELL_WINDOW_PATH,
    CONTINUOUS_TIMESERIES_PATH,
    CONTINUOUS_SUMMARY_PATH,
    CONTINUOUS_PROFILE_SUMMARY_PATH,
    CONTINUOUS_EXAMPLES_PATH,
]
if not FORCE_REBUILD_CONTINUOUS_SPELLS:
    for path in continuous_paths:
        if not path.exists():
            restore_from_drive_cache(path)
if all(path.exists() for path in continuous_paths) and not FORCE_REBUILD_CONTINUOUS_SPELLS:
    candidate_thresholds_df = pd.read_csv(CONTINUOUS_CANDIDATE_THRESHOLDS_PATH)
    catalog_spell_events_df = pd.read_csv(
        CONTINUOUS_SPELL_EVENT_PATH,
        parse_dates=[column for column in ["event_start", "event_end", "event_peak", "event_peak_jst"] if column in pd.read_csv(CONTINUOUS_SPELL_EVENT_PATH, nrows=0).columns],
    )
    catalog_spell_window_df = pd.read_csv(
        CONTINUOUS_SPELL_WINDOW_PATH,
        parse_dates=[column for column in ["spell_start", "spell_end", "first_event_peak", "last_event_peak", "first_event_peak_jst", "last_event_peak_jst", "analysis_start", "analysis_end"] if column in pd.read_csv(CONTINUOUS_SPELL_WINDOW_PATH, nrows=0).columns],
    )
    continuous_timeseries_df = pd.read_csv(
        CONTINUOUS_TIMESERIES_PATH,
        parse_dates=[column for column in ["time", "spell_start", "spell_end", "analysis_start", "analysis_end", "first_event_peak", "last_event_peak"] if column in pd.read_csv(CONTINUOUS_TIMESERIES_PATH, nrows=0).columns],
    )
    continuous_summary_df = pd.read_csv(
        CONTINUOUS_SUMMARY_PATH,
        parse_dates=[column for column in ["spell_start", "spell_end", "first_event_peak", "last_event_peak", "analysis_start", "analysis_end", "first_offshore_support_time", "first_coastal_support_time"] if column in pd.read_csv(CONTINUOUS_SUMMARY_PATH, nrows=0).columns],
    )
    continuous_profile_summary_df = pd.read_csv(CONTINUOUS_PROFILE_SUMMARY_PATH)
    continuous_examples_df = pd.read_csv(
        CONTINUOUS_EXAMPLES_PATH,
        parse_dates=[column for column in ["spell_start", "spell_end", "first_event_peak", "last_event_peak", "analysis_start", "analysis_end", "first_offshore_support_time", "first_coastal_support_time"] if column in pd.read_csv(CONTINUOUS_EXAMPLES_PATH, nrows=0).columns],
    )
    continuous_source = "restored cached continuous-spell outputs"
else:
    candidate_specs = [
        {
            "threshold_profile": "strict_baseline",
            "polygon_qflux_q": 0.60,
            "coastal_qflux_q": 0.75,
            "polygon_div_q": 0.25,
            "coastal_div_q": 0.25,
            "notes": "Current Notebook 22 default: small but strict offshore/coastal tail subsets.",
        },
        {
            "threshold_profile": "balanced_looser",
            "polygon_qflux_q": 0.55,
            "coastal_qflux_q": 0.75,
            "polygon_div_q": 0.35,
            "coastal_div_q": 0.35,
            "notes": "Slightly looser convergence thresholds while keeping offshore/coastal counts fairly balanced.",
        },
        {
            "threshold_profile": "coastal_permissive",
            "polygon_qflux_q": 0.55,
            "coastal_qflux_q": 0.60,
            "polygon_div_q": 0.35,
            "coastal_div_q": 0.35,
            "notes": "Looser coastal-moisture split to test whether later coastal evolution is being suppressed.",
        },
    ]
    candidate_threshold_rows = []
    for spec in candidate_specs:
        subset = threshold_sweep_input_df.loc[
            np.isclose(threshold_sweep_input_df["polygon_qflux_q"].astype(float), spec["polygon_qflux_q"])
            & np.isclose(threshold_sweep_input_df["coastal_qflux_q"].astype(float), spec["coastal_qflux_q"])
            & np.isclose(threshold_sweep_input_df["polygon_div_q"].astype(float), spec["polygon_div_q"])
            & np.isclose(threshold_sweep_input_df["coastal_div_q"].astype(float), spec["coastal_div_q"])
        ].copy()
        if subset.empty:
            raise RuntimeError(f"Unable to find the requested threshold profile in the Notebook 22 sweep: {spec}")
        row = subset.sort_values(["sorted_n", "offshore_n", "coastal_n"], ascending=[False, False, False]).iloc[0].to_dict()
        row.update(spec)
        candidate_threshold_rows.append(row)
    candidate_thresholds_df = pd.DataFrame(candidate_threshold_rows)[[
        "threshold_profile",
        "notes",
        "default_setting",
        "polygon_qflux_q",
        "polygon_qflux_min",
        "coastal_qflux_q",
        "coastal_qflux_split",
        "polygon_div_q",
        "polygon_div_max",
        "coastal_div_q",
        "coastal_div_max",
        "offshore_n",
        "coastal_n",
        "mixed_n",
        "weak_n",
        "sorted_n",
        "sorted_fraction",
    ]]
    if CONTINUOUS_PROFILE_FILTER is not None:
        if isinstance(CONTINUOUS_PROFILE_FILTER, str):
            profile_filter = {CONTINUOUS_PROFILE_FILTER}
        else:
            profile_filter = set(CONTINUOUS_PROFILE_FILTER)
        candidate_thresholds_df = candidate_thresholds_df.loc[
            candidate_thresholds_df["threshold_profile"].isin(profile_filter)
        ].reset_index(drop=True)
        if candidate_thresholds_df.empty:
            raise RuntimeError(
                f"CONTINUOUS_PROFILE_FILTER={CONTINUOUS_PROFILE_FILTER!r} did not match any candidate threshold profiles."
            )
    catalog_spell_input_df = event_df[[column for column in [
        "event_index",
        "event_start",
        "event_end",
        "event_peak",
        "event_peak_jst",
        "objective_regime_label",
    ] if column in event_df.columns]].copy()
    catalog_spell_events_df, catalog_spell_window_df = summarize_padded_spell_windows(
        catalog_spell_input_df,
        gap_hours=CONTINUOUS_SPELL_GAP_HOURS,
        padding_hours=CONTINUOUS_PADDING_HOURS,
        episode_prefix="catalog",
    )
    if catalog_spell_window_df.empty:
        raise RuntimeError("No catalog spells were built for the continuous-spell screen.")
    if CONTINUOUS_SPELL_LIMIT is not None:
        catalog_spell_window_df = catalog_spell_window_df.head(int(CONTINUOUS_SPELL_LIMIT)).reset_index(drop=True)
        spell_ids = set(catalog_spell_window_df["catalog_spell_id"])
        catalog_spell_events_df = catalog_spell_events_df.loc[
            catalog_spell_events_df["catalog_spell_id"].isin(spell_ids)
        ].reset_index(drop=True)
    total_spells = int(len(catalog_spell_window_df))
    print(
        f"Running continuous screen for {total_spells} catalog spells across {len(candidate_thresholds_df)} threshold profiles."
    )
    era5_runtime_ds = open_arco_era5(chunks={"time": ERA5_TIME_CHUNK})
    geometry_seed_row = catalog_spell_window_df.iloc[0]
    geometry_seed_ds = subset_era5_window(
        era5_runtime_ds,
        str(pd.Timestamp(geometry_seed_row["analysis_start"])),
        str(pd.Timestamp(geometry_seed_row["analysis_end"])),
        domain=OBJECTIVE_SUBTYPE_DOMAIN,
        variables=("specific_humidity", "vertical_velocity"),
        level=850,
    )
    region_weights = prepare_objective_region_weights(
        geometry_seed_ds.longitude,
        geometry_seed_ds.latitude,
        polygon_vertices=JPCZ_POLYGON_VERTICES,
        coastal_vertices=COASTAL_STRIP_VERTICES,
    )
    timeseries_frames = []
    continuous_summary_rows = []
    for spell_counter, (_, spell_row) in enumerate(catalog_spell_window_df.iterrows(), start=1):
        base_timeseries_df = compute_window_regional_metric_timeseries(
            era5_runtime_ds,
            start=spell_row["analysis_start"],
            end=spell_row["analysis_end"],
            region_weights=region_weights,
            domain=OBJECTIVE_SUBTYPE_DOMAIN,
            time_step_hours=CONTINUOUS_TIME_STEP_HOURS,
        )
        if base_timeseries_df.empty:
            continue
        for profile_row in candidate_thresholds_df.itertuples(index=False):
            labeled_timeseries_df = apply_continuous_regime_thresholds(
                base_timeseries_df,
                polygon_qflux_min=float(profile_row.polygon_qflux_min),
                polygon_div_max=float(profile_row.polygon_div_max),
                coastal_qflux_split=float(profile_row.coastal_qflux_split),
                coastal_div_max=float(profile_row.coastal_div_max),
                persistence_steps=CONTINUOUS_PERSISTENCE_STEPS,
            ).copy()
            labeled_timeseries_df["catalog_spell_id"] = spell_row["catalog_spell_id"]
            labeled_timeseries_df["threshold_profile"] = profile_row.threshold_profile
            labeled_timeseries_df["spell_start"] = spell_row["spell_start"]
            labeled_timeseries_df["spell_end"] = spell_row["spell_end"]
            labeled_timeseries_df["analysis_start"] = spell_row["analysis_start"]
            labeled_timeseries_df["analysis_end"] = spell_row["analysis_end"]
            labeled_timeseries_df["first_event_peak"] = spell_row["first_event_peak"]
            labeled_timeseries_df["last_event_peak"] = spell_row["last_event_peak"]
            labeled_timeseries_df["event_count"] = int(spell_row["event_count"])
            timeseries_frames.append(labeled_timeseries_df)
            summary_row = summarize_continuous_spell(
                labeled_timeseries_df,
                spell_id=str(spell_row["catalog_spell_id"]),
                threshold_profile=str(profile_row.threshold_profile),
                time_step_hours=CONTINUOUS_TIME_STEP_HOURS,
            )
            summary_row.update(
                {
                    "spell_start": spell_row["spell_start"],
                    "spell_end": spell_row["spell_end"],
                    "analysis_start": spell_row["analysis_start"],
                    "analysis_end": spell_row["analysis_end"],
                    "first_event_peak": spell_row["first_event_peak"],
                    "last_event_peak": spell_row["last_event_peak"],
                    "event_count": int(spell_row["event_count"]),
                    "spell_duration_hours": spell_row["spell_duration_hours"],
                    "analysis_window_hours": spell_row["analysis_window_hours"],
                }
            )
            continuous_summary_rows.append(summary_row)
        if spell_counter % CONTINUOUS_PROGRESS_EVERY_SPELLS == 0 or spell_counter == total_spells:
            print(f"Processed {spell_counter}/{total_spells} spells | accumulated {len(timeseries_frames)} spell-profile time series")
            if timeseries_frames and continuous_summary_rows:
                pd.concat(timeseries_frames, ignore_index=True).to_csv(CONTINUOUS_TIMESERIES_PATH, index=False)
                pd.DataFrame(continuous_summary_rows).to_csv(CONTINUOUS_SUMMARY_PATH, index=False)
                maybe_copy_to_drive(CONTINUOUS_TIMESERIES_PATH, verbose=False)
                maybe_copy_to_drive(CONTINUOUS_SUMMARY_PATH, verbose=False)
    if not timeseries_frames:
        raise RuntimeError("No continuous spell-window time series were generated.")
    continuous_timeseries_df = pd.concat(timeseries_frames, ignore_index=True)
    continuous_summary_df = pd.DataFrame(continuous_summary_rows).sort_values(["threshold_profile", "catalog_spell_id"]).reset_index(drop=True)
    profile_rows = []
    for profile_name, subset in continuous_summary_df.groupby("threshold_profile", sort=False):
        path_counts = subset["continuous_spell_regime_path"].value_counts()
        profile_rows.append(
            {
                "threshold_profile": profile_name,
                "spell_count": int(len(subset)),
                "offshore_only_spells": int(path_counts.get("offshore_only", 0)),
                "coastal_only_spells": int(path_counts.get("coastal_only", 0)),
                "offshore_to_coastal_spells": int(path_counts.get("offshore_to_coastal", 0)),
                "coastal_to_offshore_spells": int(path_counts.get("coastal_to_offshore", 0)),
                "mixed_or_oscillating_spells": int(path_counts.get("mixed_or_oscillating", 0)),
                "weak_only_spells": int(path_counts.get("weak_only", 0)),
                "directional_transition_spells": int(path_counts.get("offshore_to_coastal", 0) + path_counts.get("coastal_to_offshore", 0)),
                "median_offshore_to_coastal_lag_hours": float(subset.loc[subset["offshore_precedes_coastal"], "offshore_to_coastal_lag_hours"].median()),
                "median_coastal_to_offshore_lag_hours": float(subset.loc[subset["coastal_precedes_offshore"], "coastal_to_offshore_lag_hours"].median()),
            }
        )
    continuous_profile_summary_df = pd.DataFrame(profile_rows).sort_values(["directional_transition_spells", "offshore_only_spells", "coastal_only_spells"], ascending=[False, False, False]).reset_index(drop=True)
    continuous_examples_df = continuous_summary_df.loc[
        continuous_summary_df["continuous_spell_regime_path"].isin([
            "offshore_to_coastal",
            "coastal_to_offshore",
            "mixed_or_oscillating",
            "offshore_only",
            "coastal_only",
        ])
    ].copy()
    continuous_examples_df = continuous_examples_df.sort_values(
        ["threshold_profile", "continuous_spell_regime_path", "event_count", "analysis_window_hours"],
        ascending=[True, True, False, False],
    ).reset_index(drop=True)
    candidate_thresholds_df.to_csv(CONTINUOUS_CANDIDATE_THRESHOLDS_PATH, index=False)
    catalog_spell_events_df.to_csv(CONTINUOUS_SPELL_EVENT_PATH, index=False)
    catalog_spell_window_df.to_csv(CONTINUOUS_SPELL_WINDOW_PATH, index=False)
    continuous_timeseries_df.to_csv(CONTINUOUS_TIMESERIES_PATH, index=False)
    continuous_summary_df.to_csv(CONTINUOUS_SUMMARY_PATH, index=False)
    continuous_profile_summary_df.to_csv(CONTINUOUS_PROFILE_SUMMARY_PATH, index=False)
    continuous_examples_df.to_csv(CONTINUOUS_EXAMPLES_PATH, index=False)
    for path in continuous_paths:
        maybe_copy_to_drive(path)
    continuous_source = "computed continuous-spell outputs from ERA5 windows"
print("Continuous spell-window source:", continuous_source)
print("\nCandidate threshold finalists for continuous spell screening")
display(candidate_thresholds_df)
print("\nCatalog spell windows built from the 201-event chronology")
display(catalog_spell_window_df.head(15))
print("\nContinuous spell-profile summary")
display(continuous_profile_summary_df)
print("\nExample continuous spells")
display(continuous_examples_df.head(20))


In [ ]:

required_globals = [
    "candidate_thresholds_df",
    "continuous_timeseries_df",
    "continuous_summary_df",
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(f"Run the continuous-spell analysis cell first. Missing globals: {missing_globals}")

state_colors = {
    "offshore_jpcz_core": "#1b9e77",
    "coastal_interaction": "#d95f02",
    "mixed_transition": "#7570b3",
    "weak_or_unclear": "#bdbdbd",
}

plot_summary_df = continuous_summary_df.loc[continuous_summary_df["threshold_profile"] == CONTINUOUS_PLOT_PROFILE].copy()
if plot_summary_df.empty:
    raise RuntimeError(f"No continuous spell summaries were found for CONTINUOUS_PLOT_PROFILE={CONTINUOUS_PLOT_PROFILE!r}.")

if CONTINUOUS_PLOT_SPELL_ID is not None:
    selected_spell_row = plot_summary_df.loc[plot_summary_df["catalog_spell_id"] == CONTINUOUS_PLOT_SPELL_ID]
    if selected_spell_row.empty:
        raise RuntimeError(f"CONTINUOUS_PLOT_SPELL_ID={CONTINUOUS_PLOT_SPELL_ID!r} was not found for profile {CONTINUOUS_PLOT_PROFILE!r}.")
    selected_spell_row = selected_spell_row.iloc[0]
else:
    selected_spell_row = None
    for preferred_path in [
        "offshore_to_coastal",
        "coastal_to_offshore",
        "mixed_or_oscillating",
        "offshore_only",
        "coastal_only",
        "weak_only",
    ]:
        subset = plot_summary_df.loc[plot_summary_df["continuous_spell_regime_path"] == preferred_path]
        if not subset.empty:
            selected_spell_row = subset.sort_values(["event_count", "analysis_window_hours"], ascending=[False, False]).iloc[0]
            break
    if selected_spell_row is None:
        selected_spell_row = plot_summary_df.sort_values(["event_count", "analysis_window_hours"], ascending=[False, False]).iloc[0]

selected_spell_id = selected_spell_row["catalog_spell_id"]
selected_threshold_row = candidate_thresholds_df.loc[candidate_thresholds_df["threshold_profile"] == CONTINUOUS_PLOT_PROFILE].iloc[0]
selected_timeseries_df = continuous_timeseries_df.loc[
    (continuous_timeseries_df["threshold_profile"] == CONTINUOUS_PLOT_PROFILE)
    & (continuous_timeseries_df["catalog_spell_id"] == selected_spell_id)
].sort_values("time").copy()

if selected_timeseries_df.empty:
    raise RuntimeError(f"No continuous time series were found for spell {selected_spell_id!r} under profile {CONTINUOUS_PLOT_PROFILE!r}.")

def shade_regime_segments(ax, plot_df: pd.DataFrame):
    change_points = plot_df["continuous_regime_label"].ne(plot_df["continuous_regime_label"].shift()).cumsum()
    for _, segment_df in plot_df.groupby(change_points):
        label = str(segment_df["continuous_regime_label"].iloc[0])
        if label == "weak_or_unclear":
            continue
        ax.axvspan(segment_df["time"].iloc[0], segment_df["time"].iloc[-1], color=state_colors.get(label, "#cccccc"), alpha=0.12)

fig, axes = plt.subplots(2, 1, figsize=(13.5, 8.2), sharex=True, constrained_layout=True)

axes[0].plot(selected_timeseries_df["time"], selected_timeseries_df["polygon_qflux_850_mean"], color="#1b9e77", linewidth=2.0, label="Polygon q×(-ω)")
axes[0].plot(selected_timeseries_df["time"], selected_timeseries_df["coastal_qflux_850_mean"], color="#d95f02", linewidth=2.0, label="Coastal-only q×(-ω)")
axes[0].axhline(float(selected_threshold_row["polygon_qflux_min"]), color="#1b9e77", linestyle="--", linewidth=1.4, label="Polygon q threshold")
axes[0].axhline(float(selected_threshold_row["coastal_qflux_split"]), color="#d95f02", linestyle="--", linewidth=1.4, label="Coastal q threshold")
shade_regime_segments(axes[0], selected_timeseries_df)
axes[0].set_ylabel("Moisture proxy [1e-3 Pa s^-1]")
axes[0].set_title("Continuous spell-window moisture metrics")
axes[0].grid(alpha=0.25)
axes[0].legend(loc="upper left", ncol=2)

axes[1].plot(selected_timeseries_df["time"], selected_timeseries_df["polygon_div_925_mean"], color="#1b9e77", linewidth=2.0, label="Polygon divergence")
axes[1].plot(selected_timeseries_df["time"], selected_timeseries_df["coastal_div_925_mean"], color="#d95f02", linewidth=2.0, label="Coastal-only divergence")
axes[1].axhline(float(selected_threshold_row["polygon_div_max"]), color="#1b9e77", linestyle="--", linewidth=1.4, label="Polygon div threshold")
axes[1].axhline(float(selected_threshold_row["coastal_div_max"]), color="#d95f02", linestyle="--", linewidth=1.4, label="Coastal div threshold")
shade_regime_segments(axes[1], selected_timeseries_df)
axes[1].set_ylabel("Signed divergence [1e-5 s^-1]")
axes[1].set_xlabel("Analysis time [UTC]")
axes[1].set_title("Continuous spell-window divergence metrics")
axes[1].grid(alpha=0.25)
axes[1].legend(loc="upper left", ncol=2)

fig.suptitle(
    f"Continuous spell {selected_spell_id} | {CONTINUOUS_PLOT_PROFILE} | {selected_spell_row['continuous_spell_regime_path']}",
    fontsize=14,
)
fig.savefig(CONTINUOUS_PLOT_PATH, dpi=170, bbox_inches="tight")
plt.show()
maybe_copy_to_drive(CONTINUOUS_PLOT_PATH)

print("Selected continuous spell summary")
display(pd.DataFrame([selected_spell_row]))
print("\nFirst 20 time steps from the selected spell")
display(selected_timeseries_df.head(20))


In [ ]:
required_globals = [
    "candidate_thresholds_df",
    "catalog_spell_window_df",
    "continuous_profile_summary_df",
    "continuous_examples_df",
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(f"Run the earlier Notebook 25 cells first. Missing globals: {missing_globals}")

print("Notebook 25 summary")
print("- This notebook uses the 201-event chronology to build broader catalog spells, then pads each spell by +/- 48 h.")
print("- It computes continuous polygon and coastal-only moisture/divergence time series instead of relying on peak-event order alone.")
print("- The same three candidate threshold profiles are screened through time with a persistence rule before manual verification.")
print("- Use CONTINUOUS_SPELL_LIMIT and CONTINUOUS_PROFILE_FILTER for faster first-pass tests before full climatology runs.")
print("\nSaved outputs")
print(f"- Candidate threshold finalists: {CONTINUOUS_CANDIDATE_THRESHOLDS_PATH}")
print(f"- Catalog spell membership: {CONTINUOUS_SPELL_EVENT_PATH}")
print(f"- Padded spell windows: {CONTINUOUS_SPELL_WINDOW_PATH}")
print(f"- Continuous spell time series: {CONTINUOUS_TIMESERIES_PATH}")
print(f"- Continuous spell summary: {CONTINUOUS_SUMMARY_PATH}")
print(f"- Continuous profile summary: {CONTINUOUS_PROFILE_SUMMARY_PATH}")
print(f"- Continuous spell examples: {CONTINUOUS_EXAMPLES_PATH}")
print(f"- Selected spell plot: {CONTINUOUS_PLOT_PATH}")
